# Microsserviços — Exemplo Prático Executável

**Tópico 04 — Arquitetura de Software**

Este notebook demonstra, de forma **executável**, uma arquitetura baseada em **microsserviços**.
O cenário é um **sistema de e-commerce simplificado**, composto por três serviços independentes:

| Serviço | Porta | Responsabilidade |
|---------|-------|------------------|
| **Serviço de Produtos** | 5001 | Catálogo e controle de estoque |
| **Serviço de Pagamentos** | 5002 | Processamento (simulado) de pagamentos |
| **Serviço de Pedidos** | 5003 | Orquestra a criação de um pedido |

## Arquitetura

```
   Cliente
      │
      ▼
┌─────────────┐   HTTP   ┌─────────────┐
│  Pedidos    │─────────▶│  Produtos   │
│  :5003      │          │  :5001      │
│             │          └─────────────┘
│             │   HTTP   ┌─────────────┐
│             │─────────▶│ Pagamentos  │
└─────────────┘          │  :5002      │
                         └─────────────┘
```

Cada serviço é um processo Flask independente, rodando em uma thread separada,
comunicando-se via **HTTP/REST** — princípio fundamental de microsserviços:
**serviços autonomamente implantáveis que se comunicam por contratos leves**.

## Dependências

- `flask` — framework web para expor cada serviço via HTTP.
- `requests` — cliente HTTP usado pelo Serviço de Pedidos para chamar os outros serviços.

Já estão instaladas no ambiente virtual (`venv`). **Não é necessário reinstalar aqui.**

Se precisar reinstalar:
```python
# %pip install flask requests
```

In [1]:
import threading
import time
import requests
from flask import Flask, jsonify, request

# Silencia logs do Flask (opcional, deixa a saída mais limpa)
import logging
logging.getLogger('werkzeug').setLevel(logging.ERROR)

print("Imports OK")

Imports OK


## 1. Serviço de Produtos (porta 5001)

Responsável pelo **catálogo** e pelo **estoque**. Expõe:
- `GET /produtos/<id>` — consulta um produto
- `PUT /produtos/<id>/estoque` — decrementa o estoque após uma compra

In [2]:
app_produtos = Flask('produtos')
produtos_db = {
    1: {"nome": "Notebook", "preco": 3500.00, "estoque": 5},
    2: {"nome": "Mouse",    "preco": 150.00,  "estoque": 10},
}

@app_produtos.route('/produtos/<int:pid>', methods=['GET'])
def obter_produto(pid):
    produto = produtos_db.get(pid)
    if produto:
        return jsonify(produto), 200
    return jsonify({"erro": "Produto não encontrado"}), 404

@app_produtos.route('/produtos/<int:pid>/estoque', methods=['PUT'])
def atualizar_estoque(pid):
    dados = request.get_json()
    if pid in produtos_db:
        produtos_db[pid]['estoque'] -= dados.get('quantidade', 0)
        return jsonify({
            "mensagem": "Estoque atualizado",
            "estoque_atual": produtos_db[pid]['estoque']
        }), 200
    return jsonify({"erro": "Produto não encontrado"}), 404

def run_produtos():
    app_produtos.run(host='127.0.0.1', port=5001, debug=False, use_reloader=False)

threading.Thread(target=run_produtos, daemon=True).start()
time.sleep(2)
print("✅ Serviço de Produtos rodando em http://127.0.0.1:5001")

 * Serving Flask app 'produtos'
 * Debug mode: off
✅ Serviço de Produtos rodando em http://127.0.0.1:5001


## 2. Serviço de Pagamentos (porta 5002)

Simula um gateway de pagamento. Regra: aprova se `valor < 5000`, recusa caso contrário.

In [3]:
app_pagamentos = Flask('pagamentos')

@app_pagamentos.route('/pagamentos', methods=['POST'])
def processar_pagamento():
    dados = request.get_json()
    valor = dados.get('valor', 0)
    if valor < 5000:
        return jsonify({"status": "aprovado", "transacao_id": "txn_123"}), 200
    return jsonify({"status": "recusado", "motivo": "Valor excede limite"}), 400

def run_pagamentos():
    app_pagamentos.run(host='127.0.0.1', port=5002, debug=False, use_reloader=False)

threading.Thread(target=run_pagamentos, daemon=True).start()
time.sleep(2)
print("✅ Serviço de Pagamentos rodando em http://127.0.0.1:5002")

 * Serving Flask app 'pagamentos'
 * Debug mode: off
✅ Serviço de Pagamentos rodando em http://127.0.0.1:5002


## 3. Serviço de Pedidos (porta 5003)

Orquestra o fluxo de compra:
1. Consulta o produto no **Serviço de Produtos**.
2. Verifica estoque.
3. Chama o **Serviço de Pagamentos**.
4. Se aprovado, decrementa o estoque.

In [4]:
app_pedidos = Flask('pedidos')

PRODUTOS_URL   = "http://127.0.0.1:5001"
PAGAMENTOS_URL = "http://127.0.0.1:5002"

@app_pedidos.route('/pedidos', methods=['POST'])
def criar_pedido():
    dados = request.get_json()
    pid, qtd = dados['produto_id'], dados['quantidade']

    # 1. Consulta produto
    r = requests.get(f"{PRODUTOS_URL}/produtos/{pid}")
    if r.status_code != 200:
        return jsonify({"erro": "Produto não encontrado"}), 404
    produto = r.json()

    # 2. Verifica estoque
    if produto['estoque'] < qtd:
        return jsonify({"erro": "Estoque insuficiente"}), 400

    # 3. Calcula total e chama pagamento
    total = produto['preco'] * qtd
    r_pag = requests.post(f"{PAGAMENTOS_URL}/pagamentos", json={"valor": total})
    if r_pag.status_code != 200:
        return jsonify({
            "erro": "Pagamento recusado",
            "detalhes": r_pag.json()
        }), 400

    # 4. Atualiza estoque
    requests.put(f"{PRODUTOS_URL}/produtos/{pid}/estoque", json={"quantidade": qtd})

    return jsonify({
        "mensagem": "Pedido criado com sucesso",
        "pedido": {
            "produto": produto['nome'],
            "quantidade": qtd,
            "valor_total": total,
            "status_pagamento": r_pag.json()['status']
        }
    }), 201

def run_pedidos():
    app_pedidos.run(host='127.0.0.1', port=5003, debug=False, use_reloader=False)

threading.Thread(target=run_pedidos, daemon=True).start()
time.sleep(2)
print("✅ Serviço de Pedidos rodando em http://127.0.0.1:5003")

 * Serving Flask app 'pedidos'
 * Debug mode: off
✅ Serviço de Pedidos rodando em http://127.0.0.1:5003


## 4. Demonstração

### Cenário A — Compra aprovada
Cliente pede 1 Notebook (R$ 3.500). Valor < 5.000 → pagamento aprovado.

In [5]:
resposta = requests.post(
    "http://127.0.0.1:5003/pedidos",
    json={"produto_id": 1, "quantidade": 1}
)
print("Status:", resposta.status_code)
print("Resposta:", resposta.json())

Status: 201
Resposta: {'mensagem': 'Pedido criado com sucesso', 'pedido': {'produto': 'Notebook', 'quantidade': 1, 'status_pagamento': 'aprovado', 'valor_total': 3500.0}}


### Cenário B — Pagamento recusado
Cliente pede 2 Notebooks (R$ 7.000). Valor ≥ 5.000 → pagamento recusado.
O estoque **não** deve ser alterado.

In [6]:
# Consulta estoque antes
antes = requests.get("http://127.0.0.1:5001/produtos/1").json()
print("Estoque antes:", antes['estoque'])

resposta = requests.post(
    "http://127.0.0.1:5003/pedidos",
    json={"produto_id": 1, "quantidade": 2}
)
print("Status:", resposta.status_code)
print("Resposta:", resposta.json())

# Consulta estoque depois (deve ser o mesmo)
depois = requests.get("http://127.0.0.1:5001/produtos/1").json()
print("Estoque depois:", depois['estoque'])
assert antes['estoque'] == depois['estoque'], "Estoque mudou indevidamente!"
print("✅ Consistência mantida: estoque intacto após recusa")

Estoque antes: 4
Status: 400
Resposta: {'detalhes': {'motivo': 'Valor excede limite', 'status': 'recusado'}, 'erro': 'Pagamento recusado'}
Estoque depois: 4
✅ Consistência mantida: estoque intacto após recusa


### Cenário C — Estoque insuficiente
Cliente pede 100 Mouses (só temos 10). O Pedido nem chega a chamar o Pagamento.

In [7]:
resposta = requests.post(
    "http://127.0.0.1:5003/pedidos",
    json={"produto_id": 2, "quantidade": 100}
)
print("Status:", resposta.status_code)
print("Resposta:", resposta.json())

Status: 400
Resposta: {'erro': 'Estoque insuficiente'}


## 5. Considerações Finais

### O que foi demonstrado
- **Separação de serviços** por responsabilidade (Produtos, Pagamentos, Pedidos).
- **Comunicação via HTTP/REST** entre serviços.
- **Autonomia**: cada serviço tem seu próprio "banco" (aqui, dicionários em memória).
- **Tratamento de falhas**: pedido é abortado se pagamento for recusado ou estoque insuficiente.

### Limitações didáticas
- Persistência em memória (reinicia ao reiniciar o kernel).
- Sem service discovery, sem API Gateway, sem mensageria.
- Sem autenticação/autorização.

## Como executar

1. Crie/ative o ambiente virtual:
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   pip install flask requests jupyter ipykernel
   ```
2. Inicie o Jupyter: `jupyter notebook`
3. Selecione o kernel **Python (venv)**.
4. Rode todas as células na ordem: **Kernel → Restart & Run All**.

As três células de inicialização de serviços sobem cada serviço em uma thread
separada. Aguarde as mensagens `✅ ... rodando` antes de executar os cenários.